# Phase 0 practice: leaf disease classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavilkawdia5-ux/leaf_disease_detection/blob/main/ml/notebooks/phase0_practice.ipynb)

This is **practice only** (Phase 0 in `docs/SYSTEM_DESIGN.md`). It is not the real LeafSight model.
The real model comes in Phase 3, after you choose crops (Phase 1) and collect data (Phase 2).

You will learn:
1. **Transfer learning**: reuse a model already trained on millions of photos.
2. **Freezing**: train only the new last layer first.
3. **Fine-tuning**: then unfreeze part of the model for a little extra accuracy.
4. **Why lighting matters**: see accuracy drop on dark and shadowed photos (the LeafSight USP).
5. **Export to TFLite**: the file format the Android app will use.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.
Then run each cell with **Shift + Enter**, top to bottom.

## 1. Check the GPU

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - turn on the T4 GPU (see top of notebook)")

## 2. Load PlantVillage
About 54,000 lab photos of leaves, 38 classes (crop + disease or healthy).
The first download takes a few minutes. 85% is used for training, 15% for checking (validation).

In [ ]:
import tensorflow_datasets as tfds

(train_raw, val_raw), info = tfds.load(
    "plant_village",
    split=["train[:85%]", "train[85%:]"],
    as_supervised=True,
    with_info=True,
)
class_names = info.features["label"].names
num_classes = len(class_names)
print(num_classes, "classes, for example:", class_names[:5])

## 3. Look at some photos

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
for i, (image, label) in enumerate(train_raw.take(8)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(class_names[label].replace("___", "\n"), fontsize=8)
    plt.axis("off")
plt.show()

## 4. Prepare the images
Resize to 224x224 and put them in batches. The model does its own scaling, so pixel values stay 0-255.

In [ ]:
IMG = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def prep(image, label):
    image = tf.image.resize(image, (IMG, IMG))
    return tf.cast(image, tf.float32), label

train_ds = train_raw.map(prep, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH).prefetch(AUTOTUNE)
val_ds = val_raw.map(prep, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

## 5. Build the model
- **MobileNetV3Small**: a small, fast model made for phones, already trained on ImageNet.
- **Augmentation**: random flips, rotation, brightness and contrast while training, so the model
  learns to ignore those changes. (Phase 4 adds shadows and glare.)
- `base.trainable = False` **freezes** the pretrained part. Only the new last layer learns.

In [ ]:
augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    keras.layers.RandomContrast(0.2),
], name="augment")

base = keras.applications.MobileNetV3Small(
    input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet", pooling="avg"
)
base.trainable = False

inputs = keras.Input(shape=(IMG, IMG, 3))
x = augment(inputs)
x = base(x, training=False)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary(show_trainable=True)

## 6. Train the last layer (about 5-10 minutes)
- `accuracy` = on training photos. `val_accuracy` = on photos the model never trained on. **Watch `val_accuracy`.**

In [ ]:
EPOCHS_FROZEN = 3
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN)

## 7. Fine-tune
Unfreeze the last 30 layers of MobileNet and train with a **very small** learning rate,
so we nudge the pretrained knowledge instead of destroying it.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_fine = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_FROZEN + 2, initial_epoch=EPOCHS_FROZEN,
)

## 8. Plot the training curves

In [ ]:
acc = history.history["accuracy"] + history_fine.history["accuracy"]
val_acc = history.history["val_accuracy"] + history_fine.history["val_accuracy"]

plt.plot(acc, label="training")
plt.plot(val_acc, label="validation")
plt.axvline(EPOCHS_FROZEN - 0.5, color="gray", linestyle="--", label="start fine-tuning")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

## 9. Predictions and the "not sure" rule
LeafSight must **never guess confidently**. If the top answer's confidence is below a threshold,
the app says "not sure". (The real threshold is chosen in Phase 4.)

In [ ]:
import numpy as np

NOT_SURE_BELOW = 0.70

images, labels = next(iter(val_ds))
probs = model.predict(images, verbose=0)

plt.figure(figsize=(14, 8))
for i in range(8):
    top = int(np.argmax(probs[i]))
    conf = float(probs[i][top])
    guess = class_names[top] if conf >= NOT_SURE_BELOW else "NOT SURE"
    ok = "✓" if top == int(labels[i]) else "✗"
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(f"{ok} {guess.replace('___', chr(10))}\n{conf:.0%}", fontsize=8)
    plt.axis("off")
plt.show()

## 10. Why LeafSight exists: lighting test
PlantVillage photos are taken in a lab with perfect light. Farm photos are not.
We darken the validation photos and add a fake half-leaf shadow, then measure accuracy again.
**Expect a drop.** Fixing this drop is the whole point of LeafSight (Phases 3-5).

In [ ]:
def darken(image, label):
    return image * 0.35, label

def half_shadow(image, label):
    left = tf.fill([IMG, IMG // 2, 3], 0.4)
    right = tf.ones([IMG, IMG - IMG // 2, 3])
    return image * tf.concat([left, right], axis=1), label

base_val = val_raw.map(prep, num_parallel_calls=AUTOTUNE)
tests = {
    "good light": base_val,
    "dark": base_val.map(darken),
    "half shadow": base_val.map(half_shadow),
}
for name, ds in tests.items():
    _, acc = model.evaluate(ds.batch(BATCH).prefetch(AUTOTUNE), verbose=0)
    print(f"{name:12s} accuracy: {acc:.1%}")

## 11. Export to TensorFlow Lite
The Android app runs `.tflite` files. The real model must be **under 20 MB** (see the design doc).

In [ ]:
import os

model.export("practice_savedmodel")
converter = tf.lite.TFLiteConverter.from_saved_model("practice_savedmodel")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

with open("practice_model.tflite", "wb") as f:
    f.write(tflite_bytes)
print(f"practice_model.tflite: {os.path.getsize('practice_model.tflite') / 1e6:.1f} MB")

## Done ✓
You have ticked the last Phase 0 box: **a sample image classifier trained in Colab**.

Write down these numbers and send them to Claude Code:
- final `val_accuracy`
- the three lighting-test accuracies (good light, dark, half shadow)
- the `.tflite` size

Then move on to **Phase 1: choose your crops and diseases**.